# Python for Data Engineering -- Practice Exercises (Solved)
Student: Newana Tandukar
Day: 7

Solutions for every exercise in
`python-for-data-engineering-questions.ipynb`, in the same order and
under matching headings. Cell outputs below are the real outputs
captured during the live class session (Module 2 against the local mock
API, Module 3 against a local PostgreSQL instance) -- this notebook is
not re-executed here since those services aren't running in this
environment.

# Python for Data Engineering

### A hands-on class for the data team at **NorthStar Outfitters** (fictional online outdoor-gear retailer)

**Duration:** ~2 hours
**You should already be comfortable with:** Python data structures (lists, dicts, tuples) and basic OOP (classes, methods).
**You do NOT need any prior pandas / SQL / API experience** -- that's what today is for.

---

## The scenario

You've just joined the data engineering team. The analytics and finance teams need clean, trustworthy data to do their jobs, but right now that data is scattered:

- Order and customer records live in **CSV, Excel, and JSON files** exported from different systems.
- Two partner services -- a currency-exchange provider and a shipping carrier -- only expose their data through an **HTTP API**.
- The warehouse's shipment system writes **raw text log lines**, not tidy tables.
- Everything eventually needs to land in a **PostgreSQL database** so the BI team can query it with SQL and BI tools.

Today you'll build a small end-to-end pipeline that touches every one of those data sources. This is, in miniature, what a data engineer does every day.

## Agenda (~2 hours)

| Time | Module | What you'll do |
|---|---|---|
| 0:00 - 0:05 | Setup check | Confirm your environment is ready |
| 0:05 - 1:00 | **Module 1** -- Working with Structured Data | pandas, reading CSV/Excel/JSON, filtering, grouping, missing data |
| 1:00 - 1:40 | **Module 2** -- APIs & Semi-Structured Data | `requests`, parsing JSON, regex on log files |
| 1:40 - 2:05 | **Module 3** -- Connecting Python to Databases | SQLAlchemy/psycopg2, loading data, safe parameterized SQL |
| 2:05 - 2:10 | Wrap-up | Recap + extension ideas |

## Before you start

Make sure you've completed the **one-time setup** in `README.md` in this folder:

1. `pip install -r requirements.txt`
2. `docker compose up -d` (starts PostgreSQL for Module 3)
3. `python scripts/mock_api_server.py` running in its own terminal (powers Module 2)

If you haven't done those yet, pause and do them now -- the setup-check cell below will tell you what's missing.

In [1]:
# --- Setup check -------------------------------------------------------
# Run this first. It doesn't teach anything new -- it just confirms your
# environment is ready so you're not debugging setup issues mid-class.

import importlib
import os

import pandas as pd
import numpy as np

REQUIRED_PACKAGES = ["pandas", "numpy", "openpyxl", "requests", "sqlalchemy", "psycopg2"]
missing = [p for p in REQUIRED_PACKAGES if importlib.util.find_spec(p) is None]

print(f"pandas version:  {pd.__version__}")
print(f"numpy version:   {np.__version__}")

if missing:
    print(f"\n[MISSING PACKAGES] {missing} -- run: pip install -r requirements.txt")
else:
    print("\nAll required packages are installed.")

DATA_DIR = os.path.join("data")
expected_files = ["orders.csv", "customers.xlsx", "products.json", "shipment_logs.txt"]
present = os.listdir(DATA_DIR) if os.path.isdir(DATA_DIR) else []
missing_files = [f for f in expected_files if f not in present]

if missing_files:
    print(f"[MISSING DATA FILES] {missing_files}")
    print("  -> run: python scripts/generate_sample_data.py")
else:
    print("All data files found in data/.")

pandas version:  2.3.3
numpy version:   2.0.2

All required packages are installed.
All data files found in data/.


# Module 1: Working with Structured Data

---
# Module 1 -- Working with Structured Data

**Scenario:** Your first task is to get comfortable moving NorthStar's core business data -- orders, customers, and products -- into a shape the analytics team can actually use. This data currently lives in three different file formats, which is extremely common: different upstream systems export in whatever format is convenient for *them*, not for you.

## 1.1 Introduction to pandas: `Series` and `DataFrame`

Two objects do almost all the work in pandas:

- A **`Series`** is a single labeled column of data -- think of it as a list, but every value has an index label attached.
- A **`DataFrame`** is a table: a collection of `Series` that all share the same index, like a spreadsheet or a SQL table.

Let's see this with something small and concrete before touching real files: NorthStar's website visits for one week.

### 1. Build a Series and a DataFrame

In [2]:
import pandas as pd
import numpy as np

# A Series: one labeled column of data
daily_visits = pd.Series(
    [812, 790, 905, 1140, 1225, 1560, 980],
    index=["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"],
    name="site_visits",
)
print(daily_visits)
print()
print("Busiest day:", daily_visits.idxmax(), "with", daily_visits.max(), "visits")

Mon     812
Tue     790
Wed     905
Thu    1140
Fri    1225
Sat    1560
Sun     980
Name: site_visits, dtype: int64

Busiest day: Sat with 1560 visits


In [3]:
# A DataFrame: several related Series (columns) sharing the same index
weekly_data = {
    "site_visits": [812, 790, 905, 1140, 1225, 1560, 980],
    "orders_placed": [34, 29, 41, 55, 63, 88, 47],
    "marketing_spend_usd": [150, 150, 150, 300, 300, 500, 200],
}
weekly_summary = pd.DataFrame(weekly_data, index=daily_visits.index)
weekly_summary["conversion_rate_pct"] = round(
    weekly_summary["orders_placed"] / weekly_summary["site_visits"] * 100, 2
)
weekly_summary

,site_visits,orders_placed,marketing_spend_usd,conversion_rate_pct
Mon,812,34,150,4.19
Tue,790,29,150,3.67
Wed,905,41,150,4.53
Thu,1140,55,300,4.82
Fri,1225,63,300,5.14
Sat,1560,88,500,5.64
Sun,980,47,200,4.80


### 2. Read the CSV order export

In [4]:
# CSV -- the checkout system's order export.
# parse_dates converts order_date straight into real datetime objects instead
# of plain text, which matters the moment you want to filter or group by date.
orders_df = pd.read_csv("data/orders.csv", parse_dates=["order_date"])

print(orders_df.shape)
orders_df.head()

(600, 10)


,order_id,customer_id,product_id,quantity,unit_price,discount_pct,shipping_cost,order_date,country,order_status
0,ORD-10001,CUST-0001,P1018,3,203.01,0.00,20.67,2025-11-07 00:32:27,Australia,Shipped
1,ORD-10002,CUST-0092,P1016,2,209.30,NaN,19.45,2026-05-02 16:19:16,USA,Placed
2,ORD-10003,CUST-0093,P1033,2,62.13,0.00,4.43,2026-05-25 10:58:55,United Kingdom,Delivered
3,ORD-10004,CUST-0118,P1002,1,15.26,0.05,5.82,2025-10-12 11:43:39,Nepal,Delivered
4,ORD-10005,CUST-0038,P1005,1,209.87,0.10,10.49,2026-04-21 18:24:23,USA,Delivered


In [5]:
orders_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 600 entries, 0 to 599
Data columns (total 10 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   order_id       600 non-null    object        
 1   customer_id    594 non-null    object        
 2   product_id     600 non-null    object        
 3   quantity       600 non-null    int64         
 4   unit_price     600 non-null    float64       
 5   discount_pct   549 non-null    float64       
 6   shipping_cost  563 non-null    float64       
 7   order_date     600 non-null    datetime64[ns]
 8   country        600 non-null    object        
 9   order_status   600 non-null    object        
dtypes: datetime64[ns](1), float64(3), int64(1), object(5)
memory usage: 47.0+ KB


### 3. Read the Excel customer export

In [6]:
# Excel -- the CRM's customer export.
customers_df = pd.read_excel("data/customers.xlsx", sheet_name="customers")

print(customers_df.shape)
customers_df.head()

(150, 8)


,customer_id,first_name,last_name,email,country,signup_date,loyalty_tier,marketing_opt_in
0,CUST-0001,Priya,Sharma,priya.sharma1@example.com,Australia,2024-05-16,Bronze,True
1,CUST-0002,Ravi,Smith,ravi.smith2@example.com,Germany,2023-03-07,Bronze,True
2,CUST-0003,Ethan,Patel,ethan.patel3@example.com,USA,2024-02-12,Silver,False
3,CUST-0004,Noah,Garcia,noah.garcia4@example.com,Germany,2023-01-14,Silver,False
4,CUST-0005,Emma,Johnson,emma.johnson5@example.com,USA,2024-11-20,Bronze,True


### 4. Read and flatten the JSON product catalog

In [7]:
# JSON -- the product catalog export.
# Notice this file is NOT a flat table: each product has a nested "supplier"
# object inside it. This is extremely common with JSON. pd.json_normalize()
# flattens nested dicts into dotted column names (supplier.name, supplier.country, ...).
import json

with open("data/products.json") as f:
    products_raw = json.load(f)

print(type(products_raw), "with", len(products_raw), "items")
print(products_raw[0])  # a single product, before flattening

<class 'list'> with 40 items
{'product_id': 'P1001', 'name': 'Hiking Boots', 'category': 'Footwear', 'price': 81.0, 'stock_quantity': 358, 'supplier': {'name': 'Northern Outfitting Group', 'country': 'Canada', 'lead_time_days': 9}}


In [8]:
products_df = pd.json_normalize(products_raw)
products_df.head()

,product_id,name,category,price,stock_quantity,supplier.name,supplier.country,supplier.lead_time_days
0,P1001,Hiking Boots,Footwear,81.00,358,Northern Outfitting Group,Canada,9
1,P1002,Trail Runners,Footwear,15.26,424,Alpine Textiles Ltd.,Portugal,21
2,P1003,Camp Sandals,Footwear,75.87,289,Trailhead Industries,China,30
3,P1004,Insulated Boots,Footwear,228.54,381,Summit Gear Co.,Vietnam,18
4,P1005,Rain Jacket,Apparel,209.87,462,Trailhead Industries,China,30


### 5. Select just a few columns

In [9]:
orders_df[["order_id", "country", "order_status"]].head()

,order_id,country,order_status
0,ORD-10001,Australia,Shipped
1,ORD-10002,USA,Placed
2,ORD-10003,United Kingdom,Delivered
3,ORD-10004,Nepal,Delivered
4,ORD-10005,USA,Delivered


### 6. Filter by country

In [10]:
germany_orders = orders_df[orders_df["country"] == "Germany"]
print(f"{len(germany_orders)} orders from Germany")
germany_orders.head()

94 orders from Germany


,order_id,customer_id,product_id,quantity,unit_price,discount_pct,shipping_cost,order_date,country,order_status
16,ORD-10017,CUST-0002,P1032,2,157.75,0.05,8.45,2026-03-01 02:37:27,Germany,Shipped
19,ORD-10020,CUST-0063,P1028,2,37.99,0.05,19.53,2025-10-10 18:35:10,Germany,Delivered
29,ORD-10030,CUST-0144,P1025,2,207.37,0.00,21.32,2026-05-28 03:24:09,Germany,Delivered
31,ORD-10032,CUST-0020,P1019,3,43.77,0.20,19.38,2026-07-14 04:46:16,Germany,Shipped
32,ORD-10033,CUST-0097,P1010,1,84.13,0.20,22.91,2026-03-06 08:38:53,Germany,Delivered


### 7. Filter with combined conditions

In [11]:
us_delivered = orders_df[(orders_df["country"] == "USA") & (orders_df["order_status"] == "Delivered")]
print(f"{len(us_delivered)} delivered orders from the USA")
us_delivered.head()

103 delivered orders from the USA


,order_id,customer_id,product_id,quantity,unit_price,discount_pct,shipping_cost,order_date,country,order_status
4,ORD-10005,CUST-0038,P1005,1,209.87,0.10,10.49,2026-04-21 18:24:23,USA,Delivered
6,ORD-10007,CUST-0106,P1022,3,97.78,0.15,12.68,2025-11-18 17:16:03,USA,Delivered
11,ORD-10012,CUST-0008,P1012,1,31.62,0.00,7.40,2025-10-06 05:09:55,USA,Delivered
18,ORD-10019,CUST-0030,P1026,1,123.14,0.20,13.39,2026-04-12 03:51:15,USA,Delivered
23,ORD-10024,CUST-0078,P1018,2,203.01,0.00,NaN,2025-10-10 23:28:17,USA,Delivered


### 8. Add a computed column

In [12]:
orders_df["gross_amount"] = orders_df["quantity"] * orders_df["unit_price"]
orders_df[["order_id", "quantity", "unit_price", "gross_amount"]].head()

,order_id,quantity,unit_price,gross_amount
0,ORD-10001,3,203.01,609.03
1,ORD-10002,2,209.30,418.60
2,ORD-10003,2,62.13,124.26
3,ORD-10004,1,15.26,15.26
4,ORD-10005,1,209.87,209.87


### 9. GroupBy: total revenue per country

In [13]:
revenue_by_country = (
    orders_df.groupby("country")["gross_amount"]
    .sum()
    .sort_values(ascending=False)
    .round(2)
)
revenue_by_country

country
USA               48748.88
Germany           25478.94
United Kingdom    25307.53
Canada            20362.93
India             19443.44
Australia         12668.26
Nepal              5408.24
Name: gross_amount, dtype: float64

### 10. GroupBy with multiple stats via .agg()

In [14]:
country_status_summary = (
    orders_df.groupby(["country", "order_status"])["gross_amount"]
    .agg(order_count="count", total_revenue="sum", avg_order_value="mean")
    .round(2)
)
country_status_summary.head(10)

order_count  total_revenue  avg_order_value
country   order_status                                             
Australia Cancelled               5        1033.98           206.80
          Delivered              28        6485.57           231.63
          Placed                  3         913.05           304.35
          Returned                1         209.30           209.30
          Shipped                15        4026.36           268.42
Canada    Cancelled               6        1643.99           274.00
          Delivered              45       12441.92           276.49
          Placed                 10        2687.89           268.79
          Returned                6        1742.69           290.45
          Shipped                10        1846.44           184.64

### 11. Merge orders with customers: revenue by loyalty tier

In [15]:
orders_with_customer = orders_df.merge(
    customers_df[["customer_id", "loyalty_tier", "country"]],
    on="customer_id",
    how="left",  # keep every order even if we can't match a customer (we'll deal with that in 1.5)
)

revenue_by_tier = (
    orders_with_customer.groupby("loyalty_tier")["gross_amount"]
    .agg(order_count="count", total_revenue="sum")
    .round(2)
    .sort_values("total_revenue", ascending=False)
)
revenue_by_tier

,order_count,total_revenue
loyalty_tier,,
Bronze,350,94444.62
Silver,158,40684.19
Gold,86,21588.85


### 12. Merge orders with products: revenue by category

In [16]:
orders_with_product = orders_df.merge(
    products_df[["product_id", "category", "name"]],
    on="product_id",
    how="left",
)

revenue_by_category = (
    orders_with_product.groupby("category")["gross_amount"]
    .sum()
    .sort_values(ascending=False)
    .round(2)
)
revenue_by_category

category
Apparel         41084.29
Electronics     40369.18
Accessories     29344.26
Footwear        23608.23
Camping Gear    23012.26
Name: gross_amount, dtype: float64

### 13. Find and handle missing data

In [17]:
missing_counts = orders_df.isna().sum()
missing_counts[missing_counts > 0]

customer_id       6
discount_pct     51
shipping_cost    37
dtype: int64

In [18]:
orders_clean = orders_df.copy()

# 1. Missing discount means "no discount" -- filling with 0 is a business decision,
#    not a statistical guess.
orders_clean["discount_pct"] = orders_clean["discount_pct"].fillna(0)

# 2. Missing shipping cost: fill with the median (robust to outliers/skew).
shipping_median = orders_clean["shipping_cost"].median()
orders_clean["shipping_cost"] = orders_clean["shipping_cost"].fillna(shipping_median)
print(f"Filled missing shipping_cost with median: ${shipping_median:.2f}")

# 3. An order with no customer_id can't be attributed to anyone -- drop those rows.
before = len(orders_clean)
orders_clean = orders_clean.dropna(subset=["customer_id"])
print(f"Dropped {before - len(orders_clean)} orders with no customer_id")

print("\nRemaining missing values:")
print(orders_clean.isna().sum().sum(), "total missing cells")

Filled missing shipping_cost with median: $14.08
Dropped 6 orders with no customer_id

Remaining missing values:
0 total missing cells


### 14. Compute the net amount after cleaning

In [19]:
# Now we can safely compute the NET amount (after discount) for every order.
orders_clean["total_amount"] = round(
    orders_clean["quantity"] * orders_clean["unit_price"] * (1 - orders_clean["discount_pct"]), 2
)

orders_clean[["order_id", "quantity", "unit_price", "discount_pct", "total_amount"]].head()

,order_id,quantity,unit_price,discount_pct,total_amount
0,ORD-10001,3,203.01,0.00,609.03
1,ORD-10002,2,209.30,0.00,418.60
2,ORD-10003,2,62.13,0.00,124.26
3,ORD-10004,1,15.26,0.05,14.50
4,ORD-10005,1,209.87,0.10,188.88


# Module 2: APIs and Semi-Structured Data

`orders_clean` is now a trustworthy, analysis-ready DataFrame -- this is exactly the output of a "Module 1" pipeline stage in a real job, and it's what we'll load into PostgreSQL in Module 3.

---
# Module 2 -- Working with APIs and Semi-Structured Data

**Scenario:** Not all of NorthStar's data lives in files. Two partner services only expose their data through an HTTP API: a currency-exchange provider (finance needs to report revenue in local currencies) and a shipping-carrier tracker. On top of that, the warehouse's shipment system writes raw, semi-structured text logs, not tidy tables.

**Before running the cells below:** make sure `python scripts/mock_api_server.py` is running in its own terminal (see `README.md`). It simulates both partner services locally so this class works with no internet connection and no API keys -- but the `requests`/JSON code you write against it is identical to what you'd write against any real API.

### Setup: confirm the mock API is reachable

In [20]:
import requests

API_BASE = "http://127.0.0.1:5050"

# A quick, friendly check before we rely on the API for the rest of this module.
try:
    health = requests.get(f"{API_BASE}/health", timeout=3)
    health.raise_for_status()
    print("Mock partner API is reachable:", health.json())
except requests.exceptions.RequestException as exc:
    print("Could not reach the mock API. Start it with:")
    print("    python scripts/mock_api_server.py")
    print(f"(error: {exc})")

Mock partner API is reachable: {'status': 'ok'}


### 15. Call an HTTP API with requests

In [21]:
response = requests.get(
    f"{API_BASE}/api/v1/exchange-rates",
    params={"base": "USD"},   # becomes ?base=USD on the URL
    timeout=5,
)

print("URL requested:", response.url)
print("Status code:  ", response.status_code)

response.raise_for_status()  # raises an exception here if the call failed -- fail loudly, not silently

URL requested: http://127.0.0.1:5050/api/v1/exchange-rates?base=USD
Status code:   200


### 16. Parse the JSON response and convert currencies

In [22]:
rates_payload = response.json()
print(type(rates_payload))
rates_payload

<class 'dict'>


{'base': 'USD', 'date': '2026-08-16', 'rates': {'AUD': 1.52, 'CAD': 1.36, 'EUR': 0.92, 'GBP': 0.79, 'INR': 83.1, 'NPR': 133.5, 'USD': 1.0}}

In [23]:
country_currency = {
    "USA": "USD", "Canada": "CAD", "United Kingdom": "GBP",
    "Germany": "EUR", "Australia": "AUD", "Nepal": "NPR", "India": "INR",
}

rates = rates_payload["rates"]  # dict like {"USD": 1.0, "EUR": 0.92, ...}

revenue_report = revenue_by_country.rename("revenue_usd").to_frame()
revenue_report["local_currency"] = revenue_report.index.map(country_currency)
revenue_report["exchange_rate"] = revenue_report["local_currency"].map(rates)
revenue_report["revenue_local"] = round(revenue_report["revenue_usd"] * revenue_report["exchange_rate"], 2)

revenue_report

                revenue_usd local_currency  exchange_rate  revenue_local
country                                                                 
USA                48748.88            USD           1.00       48748.88
Germany            25478.94            EUR           0.92       23440.62
United Kingdom     25307.53            GBP           0.79       19992.95
Canada             20362.93            CAD           1.36       27693.58
India              19443.44            INR          83.10     1615749.86
Australia          12668.26            AUD           1.52       19255.76
Nepal               5408.24            NPR         133.50      722000.04

### 17. Read a raw text log file

In [24]:
with open("data/shipment_logs.txt") as f:
    log_lines = f.read().splitlines()

print(f"{len(log_lines)} log lines. First 3:\n")
for line in log_lines[:3]:
    print(line)

220 log lines. First 3:

[2026-07-01 06:40:00] INFO ShipmentService - order_id=ORD-10448 customer_id=CUST-0130 carrier=UPS tracking_number=1Z904505683 status=OUT_FOR_DELIVERY origin=Vancouver,BC dest=Kathmandu,NP weight_kg=1.22
[2026-07-01 06:52:00] INFO ShipmentService - order_id=ORD-10185 customer_id=CUST-0011 carrier=UPS tracking_number=1Z628480415 status=OUT_FOR_DELIVERY origin=Denver,CO dest=Sydney,AU weight_kg=1.39
[2026-07-01 07:24:00] INFO ShipmentService - order_id=ORD-10095 customer_id=CUST-0079 carrier=DHL Express tracking_number=1Z697892214 status=IN_TRANSIT origin=Toronto,ON dest=Toronto,ON weight_kg=0.76


### 18. Parse the log lines with a regex

In [25]:
import re

LOG_PATTERN = re.compile(
    r"^\[(?P<timestamp>[^\]]+)\]\s+(?P<level>\w+)\s+\S+ - "
    r"order_id=(?P<order_id>\S+) customer_id=(?P<customer_id>\S+) "
    r"carrier=(?P<carrier>.*?) tracking_number=(?P<tracking_number>\S+) "
    r"status=(?P<status>\S+) origin=(?P<origin>\S+) dest=(?P<dest>\S+) "
    r"weight_kg=(?P<weight_kg>[\d.]+)"
    r"(?: reason=\"(?P<reason>[^\"]*)\")?$"
)

parsed_rows = []
unmatched = 0
for line in log_lines:
    m = LOG_PATTERN.match(line)
    if m:
        parsed_rows.append(m.groupdict())
    else:
        unmatched += 1

print(f"Parsed {len(parsed_rows)} lines, {unmatched} did not match the pattern")
shipments_df = pd.DataFrame(parsed_rows)
shipments_df["weight_kg"] = shipments_df["weight_kg"].astype(float)
shipments_df.head()

Parsed 220 lines, 0 did not match the pattern


             timestamp level   order_id  ...          dest weight_kg reason
0  2026-07-01 06:40:00  INFO  ORD-10448  ...  Kathmandu,NP      1.22    NaN
1  2026-07-01 06:52:00  INFO  ORD-10185  ...     Sydney,AU      1.39    NaN
2  2026-07-01 07:24:00  INFO  ORD-10095  ...    Toronto,ON      0.76    NaN
3  2026-07-01 07:41:00  INFO  ORD-10489  ...     Denver,CO      9.97    NaN
4  2026-07-01 07:44:00  INFO  ORD-10096  ...    Bristol,UK      7.49    NaN

[5 rows x 11 columns]

### 19. Query the parsed shipments

In [26]:
delayed = shipments_df[shipments_df["status"] == "DELAYED"]
print(f"{len(delayed)} delayed shipments")
delayed[["order_id", "carrier", "origin", "dest", "reason"]]

16 delayed shipments


      order_id      carrier  ...           dest                     reason
29   ORD-10165        FedEx  ...  Manchester,UK  address correction needed
53   ORD-10516  DHL Express  ...      Sydney,AU              weather delay
64   ORD-10302  DHL Express  ...      Denver,CO  address correction needed
76   ORD-10148  DHL Express  ...      Denver,CO               customs hold
79   ORD-10382          UPS  ...  Manchester,UK  address correction needed
117  ORD-10005         USPS  ...      Austin,TX               customs hold
118  ORD-10026         USPS  ...  Manchester,UK               customs hold
130  ORD-10034         USPS  ...   Vancouver,BC              weather delay
143  ORD-10379          UPS  ...      Sydney,AU  address correction needed
154  ORD-10205        FedEx  ...      Austin,TX              weather delay
158  ORD-10260        FedEx  ...   Kathmandu,NP              weather delay
173  ORD-10356  DHL Express  ...   Melbourne,AU              weather delay
189  ORD-10239        Fed

### 20. Call the tracking API and flatten nested events

In [27]:
sample_tracking_number = delayed.iloc[0]["tracking_number"]

resp = requests.get(f"{API_BASE}/api/v1/shipments/{sample_tracking_number}", timeout=5)
resp.raise_for_status()
shipment_payload = resp.json()
shipment_payload

{'carrier': 'USPS', 'estimated_delivery': '2026-08-16', 'events': [{'event_time': '2026-08-13T05:10:01Z', 'location': 'Origin Facility', 'status': 'LABEL_CREATED'}, {'event_time': '2026-08-13T14:10:01Z', 'location': 'Regional Hub', 'status': 'IN_TRANSIT'}, {'event_time': '2026-08-13T23:10:01Z', 'location': 'Local Facility', 'status': 'OUT_FOR_DELIVERY'}], 'status': 'OUT_FOR_DELIVERY', 'tracking_number': '1Z353507570'}

In [28]:
events_df = pd.json_normalize(
    shipment_payload,
    record_path="events",
    meta=["tracking_number", "carrier", "estimated_delivery"],
)
events_df

             event_time         location  ... carrier estimated_delivery
0  2026-08-13T05:10:01Z  Origin Facility  ...    USPS         2026-08-16
1  2026-08-13T14:10:01Z     Regional Hub  ...    USPS         2026-08-16
2  2026-08-13T23:10:01Z   Local Facility  ...    USPS         2026-08-16

[3 rows x 6 columns]

### Optional / bonus: calling a real public API

In [2]:
import requests
try:
    real_resp = requests.get("https://api.frankfurter.app/latest", params={"from": "USD"}, timeout=5)
    real_resp.raise_for_status()
    print("Live rates from a real public API:")
    print(real_resp.json())
except requests.exceptions.RequestException as exc:
    print("No internet access from this environment (that's expected in many classrooms/sandboxes).")
    print(f"(error: {exc})")

Live rates from a real public API:
{'amount': 1.0, 'base': 'USD', 'date': '2026-08-14', 'rates': {'AUD': 1.412, 'BRL': 5.1762, 'CAD': 1.3875, 'CHF': 0.81179, 'CNY': 6.7413, 'CZK': 20.926, 'DKK': 6.463, 'EUR': 0.86453, 'GBP': 0.73874, 'HKD': 7.8473, 'HUF': 313.46, 'IDR': 17787, 'ILS': 2.9496, 'INR': 95.43, 'ISK': 122.94, 'JPY': 159.01, 'KRW': 1411.27, 'MXN': 16.9959, 'MYR': 4.086, 'NOK': 9.4515, 'NZD': 1.6981, 'PHP': 61.388, 'PLN': 3.7234, 'RON': 4.5325, 'SEK': 9.5089, 'SGD': 1.2781, 'THB': 33.125, 'TRY': 47.884, 'ZAR': 16.1648}}


# Module 3: Connecting Python to Databases

### 21. Connect to PostgreSQL with SQLAlchemy

In [15]:
from sqlalchemy import create_engine, text
from dotenv import load_dotenv
import os

load_dotenv()

DB_USER = "netra_neupane"
DB_PASSWORD = "Demo123" #os.getenv("POSTGRESQL_PASSWORD")
print(DB_USER, DB_PASSWORD)
DB_HOST = "localhost"
DB_PORT = 5432
DB_NAME = "northface_outfitters"

engine = create_engine(f"postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}")

with engine.connect() as conn:
    server_version = conn.execute(text("SELECT version();")).scalar()
print(server_version)

netra_neupane Demo123
PostgreSQL 16.13 on x86_64-pc-linux-musl, compiled by gcc (Alpine 15.2.0) 15.2.0, 64-bit


### 22. Create/seed a reference table and read it back

In [17]:
from sqlalchemy import text

with engine.begin() as conn:

    conn.execute(text("""
        CREATE TABLE IF NOT EXISTS carriers(
            carrier_code VARCHAR(10) PRIMARY KEY,
            carrier_name VARCHAR(100) NOT NULL,
            country VARCHAR(100)
        );
    """))

    conn.execute(text("""
        INSERT INTO carriers (
            carrier_code,
            carrier_name,
            country
        )
        VALUES
            ('FDX', 'FedEx', 'USA'),
            ('UPS', 'UPS', 'USA'),
            ('DHL', 'DHL Express', 'Germany'),
            ('USPS', 'USPS', 'USA')
        ON CONFLICT (carrier_code) DO NOTHING;
    """))

In [20]:
carriers_df = pd.read_sql("SELECT * FROM carriers ORDER BY carrier_code", engine)
carriers_df

,carrier_code,carrier_name,country
0,DHL,DHL Express,Germany
1,FDX,FedEx,USA
2,UPS,UPS,USA
3,USPS,USPS,USA


### 23. Load a cleaned DataFrame into Postgres

In [21]:
# The "Load" step: write our cleaned DataFrame into Postgres as a real table.
# if_exists="replace" is convenient in a class/dev setting; in production
# you'd more likely append, or upsert, depending on the pipeline.
orders_clean.to_sql("orders", engine, if_exists="replace", index=False)

row_count = pd.read_sql("SELECT COUNT(*) AS n FROM orders", engine).iloc[0]["n"]
print(f"Loaded {row_count} rows into the 'orders' table.")

Loaded 594 rows into the 'orders' table.


### 24. Run an aggregate SQL query from Python

In [ ]:
sql = """
    SELECT
        country,
        COUNT(*) AS n_orders,
        ROUND(SUM(total_amount)::numeric, 2) AS total_revenue
    FROM orders
    GROUP BY country
    ORDER BY total_revenue DESC
"""
revenue_by_country_sql = pd.read_sql(sql, engine)
revenue_by_country_sql

### 25. Demonstrate SQL injection

In [ ]:
def get_orders_for_customer_UNSAFE(customer_id):
    # DO NOT DO THIS -- shown only to demonstrate the vulnerability.
    query = f"SELECT order_id, order_date, order_status FROM orders WHERE customer_id = '{customer_id}'"
    print("Executing:", query)
    return pd.read_sql(query, engine)

# Looks fine for a normal, well-behaved input...
get_orders_for_customer_UNSAFE("CUST-0001")

In [ ]:
malicious_input = "CUST-0001' OR '1'='1"
result = get_orders_for_customer_UNSAFE(malicious_input)
print(f"\nReturned {len(result)} rows -- that's every order in the table, not just one customer's!")

### 26. Fix it with a parameterized SQLAlchemy query

In [ ]:
def get_orders_for_customer_SAFE(customer_id):
    query = text("SELECT order_id, order_date, order_status FROM orders WHERE customer_id = :cid")
    return pd.read_sql(query, engine, params={"cid": customer_id})

# The same malicious string is now treated as a literal, harmless value to search for --
# it matches no real customer_id, so it correctly returns nothing.
safe_result = get_orders_for_customer_SAFE(malicious_input)
print(f"Rows returned for the malicious input: {len(safe_result)}  (correct: should be 0)")

get_orders_for_customer_SAFE("CUST-0001")

### 27. Parameterize a raw psycopg2 query

In [ ]:
import psycopg2

conn = psycopg2.connect(host=DB_HOST, port=DB_PORT, dbname=DB_NAME, user=DB_USER, password=DB_PASSWORD)
cur = conn.cursor()

# %s here is psycopg2's own placeholder syntax -- NOT a Python f-string/%-format.
cur.execute(
    "SELECT order_id, order_status FROM orders WHERE customer_id = %s AND order_status = %s",
    (  "CUST-0001", "Delivered"),
)
rows = cur.fetchall()
print(rows)

cur.close()
conn.close()